In [1]:
# === Imports & chemin projet ===
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

# Ajoute le dossier racine du projet au PYTHONPATH (adapte si besoin)
root = Path.cwd()
if (root / "config").exists():
    sys.path.append(str(root))
elif (root.parent / "config").exists():
    sys.path.append(str(root.parent))

from config import settings as S
from strategies.indicators import compute_indicators
from strategies.utils import pip_size


In [4]:
CSV = Path(r"C:\Users\Elève\Downloads\ING1_GM\models_finquant\ELOE-FX\data\historical\EURUSD_1min.csv")
df = pd.read_csv(CSV)

# 2) Normaliser les noms de colonnes
df.columns = [c.strip().lower() for c in df.columns]

# 3) Trouver la colonne de date potentielle
date_candidates = ["date", "datetime", "timestamp", "time"]
date_col = next((c for c in date_candidates if c in df.columns), None)
if date_col is None:
    raise ValueError(f"Aucune colonne de date parmi {date_candidates}. Colonnes trouvées: {df.columns.tolist()}")

s = df[date_col]

# 4) Parser intelligemment (numérique = epoch en s/ms, sinon string)
if pd.api.types.is_numeric_dtype(s):
    s_num = pd.to_numeric(s, errors="coerce")
    med = float(s_num.dropna().median()) if not s_num.dropna().empty else 0.0
    unit = "ms" if med > 1e12 else "s"   # heuristique: > ~2001 en ms
    s = pd.to_datetime(s_num, unit=unit, utc=True)
else:
    s = pd.to_datetime(s, utc=True, errors="coerce")
    # si quasi tout NaT, tenter dayfirst
    if s.isna().mean() > 0.9:
        s = pd.to_datetime(df[date_col], utc=True, dayfirst=True, errors="coerce")

# 5) Index + tri
df = df.assign(date=s).dropna(subset=["date"]).set_index("date").sort_index()

# 6) Gestion fuseau: si index naïf -> localize UTC, puis convertit vers TZ de settings
if df.index.tz is None:
    df.index = df.index.tz_localize("UTC")
df.index = df.index.tz_convert(getattr(S, "TZ", "Europe/Paris"))

# 7) Ne garder que les colonnes utiles si présentes
keep_cols = [c for c in ["open", "high", "low", "close", "volume"] if c in df.columns]
if keep_cols:
    df = df[keep_cols]

print("Index tz:", df.index.tz)
print("Premières dates:", df.index[:3])
df.head()

# Indicateurs -> mêmes len/lookback que le live
df = compute_indicators(
    df,
    atr_len=getattr(S, "ATR_LEN", 14),
    rsi_len=getattr(S, "RSI_LEN", 14),
    range_lookback=getattr(S, "BREAKOUT_LOOKBACK", 30)
)

df.tail(3)


Index tz: Europe/Paris
Premières dates: DatetimeIndex(['2025-08-18 23:15:00+02:00', '2025-08-18 23:16:00+02:00',
               '2025-08-18 23:17:00+02:00'],
              dtype='datetime64[ns, Europe/Paris]', name='date', freq=None)


,open,high,low,close,volume,atr,rsi,range_high,range_low
date,,,,,,,,,
2025-08-21 00:48:00+02:00,1.16529,1.165295,1.165285,1.165290,-1.0,0.000026,43.248475,1.165455,1.165260
2025-08-21 00:49:00+02:00,1.16529,1.165300,1.165290,1.165300,-1.0,0.000025,45.157453,1.165440,1.165260
2025-08-21 00:50:00+02:00,1.16530,1.165300,1.165245,1.165265,-1.0,0.000028,40.076284,1.165440,1.165245


In [6]:
PAIR = getattr(S, "SYMBOL", "EURUSD")
RR   = float(getattr(S, "RR", 2.0))
LOOKAHEAD = 120  # minutes à scanner pour voir TP/SL

def signal_sell_row(prior_row, last_row, pair=PAIR):
    ps  = pip_size(pair)
    buf = getattr(S, "BREAKOUT_BUFFER_PIPS", 0.0) * ps
    use_hl = getattr(S, "BREAKOUT_USE_HIGH_LOW", True)

    dn_break = (last_row.low  < prior_row["range_low"] - buf) if use_hl else (last_row.close < prior_row["range_low"] - buf)
    rsi_ok   = last_row.rsi <= (100 - getattr(S, "RSI_MIN", 55))
    return bool(dn_break and rsi_ok)

def label_sell_tp_first(df, i, pair=PAIR, rr=RR, lookahead=LOOKAHEAD):
    """Retourne 1 si TP SELL touché avant SL, 0 si SL avant TP, None si non résolu."""
    if i >= len(df) - 1:
        return None

    entry = float(df["close"].iloc[i])
    # ATR: lire proprement un scalaire ; fallback propre = np.nan
    atr_i = float(df["atr"].iloc[i]) if "atr" in df.columns else np.nan

    ps = pip_size(pair)
    ATR_K = float(getattr(S, "ATR_K", 1.5))
    MIN_STOP_PIPS = float(getattr(S, "MIN_STOP_PIPS", 8.0))

    stop_pips = MIN_STOP_PIPS if (np.isnan(atr_i) or atr_i <= 0) else max((atr_i / ps) * ATR_K, MIN_STOP_PIPS)
    stop_dist = stop_pips * ps

    sl = entry + stop_dist         # SELL: stop au-dessus
    tp = entry - rr * stop_dist    # SELL: take en dessous

    max_j = min(lookahead, len(df) - i - 1)
    for j in range(1, max_j + 1):
        hi = float(df["high"].iloc[i + j])
        lo = float(df["low"].iloc[i + j])
        if lo <= tp:
            return 1   # TP d’abord
        if hi >= sl:
            return 0   # SL d’abord
    return None

# --- Construire l’échantillon SELL ---
rows, labels = [], []
for i in range(1, len(df)):
    prior, last = df.iloc[i-1], df.iloc[i]
    if signal_sell_row(prior, last, PAIR):
        y = label_sell_tp_first(df, i, PAIR, RR, LOOKAHEAD)
        if y is not None:
            rows.append(i)
            labels.append(int(y))

print(f"Nb signaux SELL résolus: {len(rows)} (TP={sum(labels)}, SL={len(labels)-sum(labels)})")




Nb signaux SELL résolus: 74 (TP=24, SL=50)


In [7]:
# Essaie de réutiliser le même "build_features" que pour BUY
try:
    from models.proba_model import build_features
    X_full = build_features(df)
except Exception:
    # fallback simple
    X_full = pd.DataFrame(index=df.index)
    X_full["ret1"] = df["close"].pct_change()
    X_full["atr"]  = df.get("atr", pd.Series(index=df.index))
    X_full["rsi"]  = df.get("rsi", pd.Series(index=df.index))
    X_full["rng"]  = df["high"] - df["low"]

# Échantillon SELL uniquement
X = X_full.loc[df.index[rows]].replace([np.inf, -np.inf], np.nan).fillna(0)
y = np.array(labels, dtype=int)

X.shape, y.mean()  # taille et taux de TP


((74, 15), np.float64(0.32432432432432434))

In [9]:

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score
import numpy as np

def pick_threshold(p, y_true, rr=RR):
    thrs = np.linspace(0.05, 0.95, 19)
    best_ev, best_t = -1e9, 0.5
    for t in thrs:
        sel = p >= t
        n = sel.sum()
        if n == 0:
            continue
        nb_tp = int((y_true[sel] == 1).sum())
        nb_sl = int((y_true[sel] == 0).sum())
        ev = (nb_tp * rr - nb_sl * 1.0) / n
        if ev > best_ev:
            best_ev, best_t = ev, t
    return best_t, best_ev

candidates = {
    "LR": LogisticRegression(max_iter=300, class_weight="balanced"),
    "RF": RandomForestClassifier(
        n_estimators=300, max_depth=None, min_samples_leaf=5, n_jobs=-1,
        class_weight="balanced_subsample"
    ),
}

tscv = TimeSeriesSplit(n_splits=5)

best = {"name": None, "auc": -np.inf, "thr": 0.5, "ev": -1e9, "model": None}
valid_folds = 0

for name, model in candidates.items():
    aucs, thrs, evs = [], [], []
    for fold, (tr, va) in enumerate(tscv.split(X)):
        ytr, yva = y[tr], y[va]
        if len(np.unique(ytr)) < 2 or len(np.unique(yva)) < 2:
            print(f"[SKIP] {name} fold {fold}: classes train={np.unique(ytr)} val={np.unique(yva)}")
            continue
        Xtr, Xva = X.iloc[tr].values, X.iloc[va].values
        model.fit(Xtr, ytr)
        p = model.predict_proba(Xva)[:, 1]
        auc = roc_auc_score(yva, p)
        thr, ev = pick_threshold(p, yva, rr=RR)
        aucs.append(auc); thrs.append(thr); evs.append(ev)
        valid_folds += 1
    if aucs:
        auc_m, thr_m, ev_m = float(np.mean(aucs)), float(np.median(thrs)), float(np.mean(evs))
        print(f"{name}: AUC={auc_m:.3f}  thr(EV-opt)~{thr_m:.3f}  EV_val={ev_m:.3f}")
        if auc_m > best["auc"] or (np.isclose(auc_m, best["auc"]) and ev_m > best["ev"]):
            best.update(name=name, auc=auc_m, thr=thr_m, ev=ev_m, model=model)

# Fallback si toutes les folds ont été skip
if best["model"] is None or valid_folds == 0:
    print("[WARN] Pas de fold valide (mono-classe). Fallback 80/20 temporel.")
    cut = int(len(X) * 0.8)
    Xtr, ytr = X.iloc[:cut].values, y[:cut]
    Xva, yva = X.iloc[cut:].values, y[cut:]
    if len(np.unique(ytr)) < 2 or len(np.unique(yva)) < 2:
        raise SystemExit(
            "Toujours mono-classe après fallback. "
            "=> Augmente l'historique, relâche RSI/lookback/buffer, ou augmente LOOKAHEAD."
        )
    # Essaie LR par défaut en fallback
    model = LogisticRegression(max_iter=300, class_weight="balanced")
    model.fit(Xtr, ytr)
    p = model.predict_proba(Xva)[:, 1]
    auc = roc_auc_score(yva, p)
    thr, ev = pick_threshold(p, yva, rr=RR)
    best.update(name="LR", auc=float(auc), thr=float(thr), ev=float(ev), model=model)

best


[SKIP] LR fold 0: classes train=[1] val=[0 1]
[SKIP] LR fold 1: classes train=[0 1] val=[0]
[SKIP] LR fold 2: classes train=[0 1] val=[0]
[SKIP] LR fold 4: classes train=[0 1] val=[0]
LR: AUC=0.630  thr(EV-opt)~0.050  EV_val=0.286
[SKIP] RF fold 0: classes train=[1] val=[0 1]
[SKIP] RF fold 1: classes train=[0 1] val=[0]
[SKIP] RF fold 2: classes train=[0 1] val=[0]
[SKIP] RF fold 4: classes train=[0 1] val=[0]
RF: AUC=0.222  thr(EV-opt)~0.050  EV_val=-0.250


{'name': 'LR',
 'auc': 0.6296296296296297,
 'thr': 0.05,
 'ev': 0.2857142857142857,
 'model': LogisticRegression(class_weight='balanced', max_iter=300)}

In [12]:
final = LogisticRegression(max_iter=300, class_weight="balanced")
final.fit(X.values, y)

MODEL_SELL = {
    "model": final,
    "features": list(X.columns),
    "threshold": float(best["thr"]),   # ~0.05 d’après ta CV
    "meta": {
        "pair": getattr(S, "SYMBOL", "EURUSD"),
        "rr": float(getattr(S, "RR", 2.0)),
        "lookahead_min": 120,
        "atr_k": float(getattr(S, "ATR_K", 1.5)),
        "min_stop_pips": float(getattr(S, "MIN_STOP_PIPS", 8)),
        "rsi_min": float(getattr(S, "RSI_MIN", 55)),
        "breakout_lookback": float(getattr(S, "BREAKOUT_LOOKBACK", 30)),
        "buffer_pips": float(getattr(S, "BREAKOUT_BUFFER_PIPS", 0.0)),
        "use_high_low": bool(getattr(S, "BREAKOUT_USE_HIGH_LOW", True)),
        "auc_cv": float(best["auc"]),
        "ev_val_mean": float(best["ev"]),
        "note": "SELL model trained with TimeSeriesSplit; threshold from EV-opt."
    }
}

Path("models").mkdir(parents=True, exist_ok=True)
out_path = Path(getattr(S, "SELL_MODEL_PATH", "../models/model_eurusd_sell.pkl"))
joblib.dump(MODEL_SELL, out_path)
print(f"✅ Modèle SELL sauvegardé -> {out_path}")
print(f"AUC(cv)={best['auc']:.3f}  threshold={MODEL_SELL['threshold']:.3f}  EV_val≈{best['ev']:.3f} R/trade")

✅ Modèle SELL sauvegardé -> ..\models\model_eurusd_sell.pkl
AUC(cv)=0.630  threshold=0.050  EV_val≈0.286 R/trade
